# Lab 08 - Decision Tree Classification
## Heart Disease Prediction using Decision Tree Classifier

**Objective:** To implement a Decision Tree Classifier for predicting heart disease using the Cleveland Heart Disease dataset.

### 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

### 2. Load the Dataset

In [ ]:
# Define column names for the Cleveland Heart Disease dataset
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

# Load the dataset
df = pd.read_csv('processed.cleveland.data', header=None, names=columns, na_values='?')

print("Shape of dataset:", df.shape)
df.head()

### 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info about the dataset
print("Dataset Info:")
print(df.info())
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# Convert target to binary classification (0 = No Disease, 1 = Disease)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print("Target distribution:")
print(df['target'].value_counts())

# Plot target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Heart Disease Distribution')
plt.xlabel('0 = No Disease, 1 = Disease')
plt.ylabel('Count')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

### 4. Data Preprocessing

In [ ]:
# Handle missing values - fill with median
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
df.fillna(df.median(), inplace=True)

print("Missing values after handling:")
print(df.isnull().sum())

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

### 5. Build Decision Tree Model

In [ ]:
# Create and train the Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
dt_classifier.fit(X_train, y_train)

print("Decision Tree trained successfully!")
print("Tree depth:", dt_classifier.get_depth())
print("Number of leaves:", dt_classifier.get_n_leaves())

In [ ]:
# Visualize the Decision Tree
plt.figure(figsize=(20, 10))
plot_tree(dt_classifier, feature_names=X.columns, class_names=['No Disease', 'Disease'],
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree for Heart Disease Prediction')
plt.tight_layout()
plt.show()

### 6. Model Evaluation

In [ ]:
# Predictions
y_pred = dt_classifier.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 7. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.Series(dt_classifier.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_importance.plot(kind='barh', color='teal')
plt.title('Feature Importance - Decision Tree')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### 8. Hyperparameter Tuning - Effect of max_depth

In [ ]:
# Test different max_depth values
depths = range(1, 15)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, 'b-o', label='Training Accuracy')
plt.plot(depths, test_scores, 'r-o', label='Testing Accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Effect of Max Depth on Model Performance')
plt.legend()
plt.grid(True)
plt.show()

best_depth = depths[np.argmax(test_scores)]
print(f"Best max_depth: {best_depth} with test accuracy: {max(test_scores):.4f}")

### 9. Conclusion

In this lab, we implemented a Decision Tree Classifier on the Cleveland Heart Disease dataset. The model was trained to predict whether a patient has heart disease based on 13 clinical features. We performed EDA, handled missing values, trained the model, evaluated its performance using accuracy, classification report, and confusion matrix, and analyzed feature importances. We also explored the effect of the max_depth hyperparameter on model performance.